# 02 Preprocessing

Prepare the raw ASHRAE Great Energy Predictor III data for exploratory analysis and machine learning.

## Setup

In [4]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## Load Raw Data

In [5]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

raw_data_dir = project_root / "data" / "raw"
interim_data_dir = project_root / "data" / "interim"
processed_data_dir = project_root / "data" / "processed"

train_path = raw_data_dir / "train.csv"
building_path = raw_data_dir / "building_metadata.csv"
weather_train_path = raw_data_dir / "weather_train.csv"

In [6]:
train = pd.read_csv(train_path)
building_metadata = pd.read_csv(building_path)
weather_train = pd.read_csv(weather_train_path)

## Filter Electricity Meter Records

The project focuses on electricity consumption, represented by `meter = 0`.

In [7]:
electricity_train = train[train["meter"] == 0].copy()

electricity_train.shape

(12060910, 4)

## Convert Timestamp Columns

In [8]:
electricity_train["timestamp"] = pd.to_datetime(electricity_train["timestamp"])
weather_train["timestamp"] = pd.to_datetime(weather_train["timestamp"])

## Merge Datasets

Merge electricity records with building metadata, then merge weather observations using `site_id` and `timestamp`.

In [9]:
electricity_merged = electricity_train.merge(
    building_metadata,
    on="building_id",
    how="left",
)

electricity_merged = electricity_merged.merge(
    weather_train,
    on=["site_id", "timestamp"],
    how="left",
)

electricity_merged.shape

(12060910, 16)

## Check Missing Values

In [10]:
missing_values = (
    electricity_merged.isna()
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

missing_values.columns = ["column", "missing_count"]
missing_values["missing_pct"] = (
    missing_values["missing_count"] / len(electricity_merged) * 100
).round(2)

missing_values

,column,missing_count,missing_pct
0,floor_count,9096083,75.42
1,year_built,6470035,53.64
2,cloud_coverage,5329652,44.19
3,precip_depth_1_hr,2513679,20.84
4,sea_level_pressure,1018383,8.44
5,wind_direction,678715,5.63
6,wind_speed,66795,0.55
7,dew_temperature,49091,0.41
8,air_temperature,47325,0.39
9,building_id,0,0.00


## Handle Missing Values

Use simple, explainable baseline choices: drop highly incomplete `floor_count`, fill `year_built` with its median, fill missing categories as `Unknown`, fill precipitation with `0`, and fill remaining weather values with site-level medians.

In [11]:
cleaned_data = electricity_merged.copy()

In [12]:
cleaned_data = cleaned_data.drop(columns=["floor_count"])

In [13]:
cleaned_data["year_built"] = cleaned_data["year_built"].fillna(
    cleaned_data["year_built"].median()
)

In [14]:
cleaned_data["primary_use"] = cleaned_data["primary_use"].fillna("Unknown")

In [15]:
cleaned_data["precip_depth_1_hr"] = cleaned_data["precip_depth_1_hr"].fillna(0)

In [16]:
weather_columns = [
    "air_temperature",
    "cloud_coverage",
    "dew_temperature",
    "sea_level_pressure",
    "wind_direction",
    "wind_speed",
]

for column in weather_columns:
    cleaned_data[column] = cleaned_data.groupby("site_id")[column].transform(
        lambda values: values.fillna(values.median())
    )

    cleaned_data[column] = cleaned_data[column].fillna(
        cleaned_data[column].median()
    )

## Verify Missing Values

In [17]:
cleaned_data.isna().sum().sort_values(ascending=False)

building_id           0
meter                 0
timestamp             0
meter_reading         0
site_id               0
primary_use           0
square_feet           0
year_built            0
air_temperature       0
cloud_coverage        0
dew_temperature       0
precip_depth_1_hr     0
sea_level_pressure    0
wind_direction        0
wind_speed            0
dtype: int64

## Create Time Features

In [21]:
cleaned_data["hour"] = cleaned_data["timestamp"].dt.hour
cleaned_data["day_of_week"] = cleaned_data["timestamp"].dt.dayofweek
cleaned_data["month"] = cleaned_data["timestamp"].dt.month
cleaned_data["is_weekend"] = cleaned_data["day_of_week"].isin([5, 6]).astype(int)
cleaned_data["date"] = cleaned_data["timestamp"].dt.date

cleaned_data.head(1000)

,building_id,meter,timestamp,meter_reading,site_id,primary_use,square_feet,year_built,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed,hour,day_of_week,month,is_weekend,date
0,0,0,2016-01-01,0.00,0,Education,7432,2008.0,25.0,6.0,20.0,0.0,1019.7,0.0,0.0,0,4,1,0,2016-01-01
1,1,0,2016-01-01,0.00,0,Education,2720,2004.0,25.0,6.0,20.0,0.0,1019.7,0.0,0.0,0,4,1,0,2016-01-01
2,2,0,2016-01-01,0.00,0,Education,5376,1991.0,25.0,6.0,20.0,0.0,1019.7,0.0,0.0,0,4,1,0,2016-01-01
3,3,0,2016-01-01,0.00,0,Education,23685,2002.0,25.0,6.0,20.0,0.0,1019.7,0.0,0.0,0,4,1,0,2016-01-01
4,4,0,2016-01-01,0.00,0,Education,116607,1975.0,25.0,6.0,20.0,0.0,1019.7,0.0,0.0,0,4,1,0,2016-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1029,0,2016-01-01,246.00,11,Education,152559,1970.0,-1.8,2.0,-3.2,0.0,1016.0,280.0,1.5,0,4,1,0,2016-01-01
996,1030,0,2016-01-01,50.00,11,Education,68030,1970.0,-1.8,2.0,-3.2,0.0,1016.0,280.0,1.5,0,4,1,0,2016-01-01
997,1031,0,2016-01-01,207.50,11,Education,93206,1970.0,-1.8,2.0,-3.2,0.0,1016.0,280.0,1.5,0,4,1,0,2016-01-01
998,1032,0,2016-01-01,206.00,11,Education,127632,1970.0,-1.8,2.0,-3.2,0.0,1016.0,280.0,1.5,0,4,1,0,2016-01-01


## Save Processed Data

In [19]:
interim_data_dir.mkdir(parents=True, exist_ok=True)
processed_data_dir.mkdir(parents=True, exist_ok=True)

electricity_merged.to_csv(
    interim_data_dir / "electricity_merged.csv",
    index=False,
)

cleaned_data.to_csv(
    processed_data_dir / "electricity_cleaned.csv",
    index=False,
)

## Final Check

In [20]:
print("Merged electricity data shape:", electricity_merged.shape)
print("Cleaned data shape:", cleaned_data.shape)
print("Processed file saved to:", processed_data_dir / "electricity_cleaned.csv")

Merged electricity data shape: (12060910, 16)
Cleaned data shape: (12060910, 20)
Processed file saved to: /Users/srs/Desktop/PROJECTS/commercial-building-energy-prediction/data/processed/electricity_cleaned.csv
